# Can a machine hear an earthquake?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F13_machine_hears.ipynb).

A seismometer never stops. It writes down how the ground is moving a hundred times a second, day
and night, and nearly all of what it writes is traffic, wind and surf. Buried in that are the
earthquakes, almost all of them far too small to feel.

Finding them is one job. Timing them is a harder one, and it is the one that matters. What a
seismologist needs from a recording is the *instant* the first wave arrived, because the gap
between the fast P wave and the slower S wave gives the distance to the earthquake, and distances
from three stations give its location. For most of the history of the subject that instant was
marked by a person, by eye.

Today you get 2,500 real recordings from northern California, each already marked by an analyst.
You will watch the standard automatic picker miss, and then train a small network to find the
instant for you.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, then export the notebook as a PDF and upload that.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say what a seismogram contains and why the P and S arrival times are the
quantity a seismologist actually wants. Show that loudness answers *whether* an earthquake
happened but not *when* it started, and say how well the classical automatic picker does on real
data before anything is learned from it.

**The skills.** Slide a pattern-detector along a signal with `np.convolve`. Build a neural
network in PyTorch out of `nn.Conv1d`, `nn.ReLU` and `nn.Upsample`, give it a loss to make small,
train it with gradient descent for a fixed number of epochs, and read its learning curve to see
when more training stops buying anything.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.linear_model import LinearRegression

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

WAVEFORMS = ("https://github.com/AI4EPS/EPS88_PyEarth/releases/download/data-v1/phasenet_ncedc.npz")


def load():
    """Read the waveform file, downloading it from the course release the first time."""
    try:
        return np.load("phasenet_ncedc.npz")
    except FileNotFoundError:
        torch.hub.download_url_to_file(WAVEFORMS, "phasenet_ncedc.npz", progress=False)
        return np.load("phasenet_ncedc.npz")


# 42 MB of waveforms — too big to keep beside the notebook, so it arrives from a release of the
# course repository rather than from the repository itself, and is kept once it has arrived.
data = load()
waveform = np.clip(data["waveform"], -10, 10)   # instrument spikes, cut off at 10 times normal
p_index = data["p_index"].astype(int)           # sample number of the analyst's P pick
s_index = data["s_index"].astype(int)           # sample number of the analyst's S pick
distance_km = data["distance_km"]
event_id = data["event_id"]
station = data["station"]
magnitude = data["magnitude"]
snr = data["snr"]                               # how many times louder the earthquake is than
SAMPLE_RATE = 100                               # the background, on that recording

print("recordings:      ", waveform.shape)
print("earthquakes:     ", len(np.unique(event_id)), " stations:", len(np.unique(station)))
print("magnitudes:      ", round(magnitude.min(), 2), "to", round(magnitude.max(), 2),
      " median", round(np.median(magnitude), 2))
print("distance (km):   ", round(np.median(distance_km), 1), "median,",
      round(distance_km.max(), 1), "furthest")
print("P arrives between", round(p_index.min() / SAMPLE_RATE, 2), "and",
      round(p_index.max() / SAMPLE_RATE, 2), "seconds in")

## Twenty seconds of ground motion

`waveform` holds 2,500 recordings. Each one is a small array of its own: **3 rows**,
because a seismometer measures the ground moving east, north and up-down at the same time, and
**2,048 columns**, one per sample. At 100 samples a second that is
20.48 seconds of shaking per recording.

These come from the Northern California Earthquake Data Center, whose analysts marked the P and
the S on every one by hand. `p_index` and `s_index` are those marks, stored as sample numbers, so
sample 1024 means 10.24 seconds into the
window. Each row has been divided by its own typical size, so a "1" in `waveform` means *one
typical wiggle for this instrument on this recording*, not a fixed number of nanometres —
loudness here is always relative to the same trace's own background.

One thing was done deliberately when the file was built: the window was cut at a **random** offset
before each P, so the arrival lands anywhere from 3.07 to 9.21 seconds in.
Nothing here can score well by always answering "sample 1024".

Look at one. The three rows are drawn on the same axes, and the two vertical lines are the
analyst's marks. Before the first line there is nothing but background. The P is the first
arrival; the S, which follows it, shakes harder and is the wave that does the damage.

In [ ]:
seconds = np.arange(2048) / SAMPLE_RATE
example = waveform[4]

for row, name in zip(example, ["east", "north", "up-down"]):
    plt.plot(seconds, row, lw=0.6, label=name)
plt.axvline(p_index[4] / SAMPLE_RATE, color="k")
plt.axvline(s_index[4] / SAMPLE_RATE, color="k")
plt.xlabel("time (s)")
plt.ylabel("ground motion (in units of this trace's own size)")
plt.title("station NC.BBGB, magnitude 0.94 — 1 of "
          "2,500 recordings; the lines are P and S")
plt.legend()
plt.show()

### ✏️ Your turn 1

Draw a different one. Choose any trace number you like between 0 and 2499, plot
just its **up-down** row (that is `waveform[my_trace][2]`) against `seconds`, mark the P and the S
with `plt.axvline`, and label both axes.

Then print four things in seconds: the P time, the S time, the gap between them, and the moment
of the **largest swing** on that row. `np.abs(row).argmax()` gives the sample number of the
largest value, and dividing a sample number by `SAMPLE_RATE` turns it into seconds.

**Use these names**, because the self-check looks for them: `my_trace` for the trace number,
`sp_seconds` for the gap, and `loudest_second` for the moment of the largest swing.

In [ ]:
# ← your answer here


assert sp_seconds > 0, "the S always arrives after the P, so the gap must be positive"
print("✓ one trace — trace", my_trace, "has its S", round(sp_seconds, 2),
      "s after its P, and its biggest swing at", loudest_second, "s")

That gap is the whole reason anyone cares about the exact arrival time. The P and the S leave the
earthquake together and travel at different speeds, so the further you are from it, the further
apart they arrive — the gap is a distance measurement, taken at a single station.

Which means we can check it. Fit a straight line through the origin (through the origin because a
station standing on top of the earthquake must see a gap of zero) and the slope is how many
kilometres each second of gap is worth.

In [ ]:
sp_all = (s_index - p_index) / SAMPLE_RATE

line = LinearRegression(fit_intercept=False)
line.fit(sp_all.reshape(-1, 1), distance_km)

ends = np.array([0, sp_all.max()]).reshape(-1, 1)   # a straight line needs only its two ends

plt.scatter(sp_all, distance_km, s=3, alpha=0.3)
plt.plot(ends, line.predict(ends), color="C1")
plt.xlabel("S minus P (s)")
plt.ylabel("distance to the earthquake (km)")
plt.title("2,500 recordings: the gap is a distance measurement")
plt.show()

print("kilometres per second of gap:", round(line.coef_[0], 2))
print("R squared:", round(line.score(sp_all.reshape(-1, 1), distance_km), 3))

## Was there an earthquake at all?

Before *when*, the easier question: **did anything happen?** Cut two windows out of every
recording, each 2.56 seconds long — one ending just before the analyst's P, one
starting just after it. The first contains only background noise. The second contains an
earthquake. A machine that can tell those apart is an earthquake detector.

We are going to be scoring things from here to the end of the notebook, so build the held-out set
first — and build it by **earthquake**, not by recording. Several stations recorded the same
event; if some of those recordings go into training and the rest into testing, the test is not
held out at all. That is leakage, and grouping by `event_id` is the fix.

In [ ]:
rng = np.random.default_rng(0)
events = np.unique(event_id)
rng.shuffle(events)
train_events = events[:int(0.7 * len(events))]
is_train = np.isin(event_id, train_events)      # True for a recording of a training earthquake

print("training on", is_train.sum(), "recordings from", len(train_events), "earthquakes")
print("testing on ", (~is_train).sum(), "recordings from",
      len(events) - len(train_events), "earthquakes")

In [ ]:
WINDOW = 256
quiet = np.zeros((len(waveform), 3, WINDOW), dtype="float32")
shaken = np.zeros((len(waveform), 3, WINDOW), dtype="float32")
for i in range(len(waveform)):
    quiet[i] = waveform[i][:, p_index[i] - WINDOW - 6:p_index[i] - 6]
    shaken[i] = waveform[i][:, p_index[i] + 6:p_index[i] + 6 + WINDOW]

loud_quiet = np.abs(quiet).max(axis=(1, 2))     # biggest swing in each before-window
loud_shaken = np.abs(shaken).max(axis=(1, 2))   # biggest swing in each after-window

print("typical biggest swing before the P:", round(np.median(loud_quiet[is_train]), 2))
print("typical biggest swing after the P: ", round(np.median(loud_shaken[is_train]), 2))

Those two medians came from the training recordings only, so we are allowed to look at them.
Now the rule. **Write the dumbest rule you can, first. Any model that cannot beat it is
decoration.** The dumbest rule here is one number: call it an earthquake when the biggest swing
in the window is above 2.5, which sits between the two typical values you just
printed.

### ✏️ Your turn 2

Score the dumb rule on the held-out recordings only.

A before-window is correct when its biggest swing is **below** 2.5; an after-window
is correct when its biggest swing is **at or above** 2.5. Count both, add them up,
and divide by the number of windows you scored — remember every held-out recording gives you two
windows, one of each kind.

**Use these names**, because the self-check looks for them: `detector_accuracy`.

In [ ]:
# ← your answer here


assert detector_accuracy <= 1, "an accuracy is a fraction of the windows, so it cannot exceed 1"
print("✓ the one-number rule — it is right on",
      round(100 * detector_accuracy, 1), "% of held-out windows")

So *detection* barely needs us. One `if` statement, one number, and most of the work is done —
which is exactly why a neural network for this task would be decoration.

The interesting question was never whether the ground shook. It is **when it started**, and your
own trace gave you one data point on that: compare the moment of its largest swing with the moment
of its P. Here is the same comparison, over all 2,500 recordings at once.

In [ ]:
strength = np.abs(waveform).max(axis=1)     # one number per sample: the biggest of the three rows
loudest = strength.argmax(axis=1)           # sample number of the biggest swing in each recording
near_p = np.abs(loudest - p_index) <= 50    # 50 samples is half a second
near_s = np.abs(loudest - s_index) <= 50

print("loudest sample is within 0.5 s of the P:", round(near_p.mean(), 3))
print("loudest sample is within 0.5 s of the S:", round(near_s.mean(), 3))

plt.hist((loudest - p_index) / SAMPLE_RATE, bins=60)
plt.axvline(0, color="k")
plt.xlabel("loudest sample minus P arrival (s)")
plt.ylabel("number of recordings")
plt.title("where the biggest swing sits, 2,500 recordings; 0 is the P")
plt.show()

The tallest bar does sit against the line — but it holds only about a sixth of the recordings,
and the rest are strung out for seconds afterwards, with a scatter to the left as well, on
recordings where the loudest thing in the window happened before the earthquake got there at all.
16.9% of the loudest samples are within half a second of the P;
53.5% are within half a second of the **S**. Loudness is a fine answer to *did
something happen*. It is close to useless as an answer to *when did it start*, because the loudest
moment is usually a different wave arriving.

Whatever finds the P has to work on the **shape** of the signal — the moment a quiet trace stops
being quiet — and not on how big it gets.

## The classical automatic picker

Seismology has had an automatic answer to this for decades, and it is two averages and a
division. Take the average power over a **short** window ending at the current sample, take it
again over a **long** window ending at the same sample, and divide. In background noise the two
averages are the same and the ratio sits near 1. The instant a wave arrives, the short window
fills with the new energy while the long window is still mostly old quiet — so the ratio jumps.
Trigger the first time it crosses some level, and that is your pick.

It is called STA/LTA, for short-term average over long-term average. Three numbers to choose: how
short, how long, and how big a jump counts. The ratio below is left at zero until there is a full
long window behind it to average over — five seconds, at this setting.

In [ ]:
def sta_lta(trace, short, long):
    """How much louder the last `short` samples are than the last `long` samples."""
    power = trace ** 2
    ratio = np.zeros(len(power))
    for i in range(long, len(power)):
        ratio[i] = power[i - short:i].mean() / power[i - long:i].mean()
    return ratio


def first_trigger(ratio, threshold):
    """The first sample where the ratio crosses the threshold, or None if it never does."""
    above = np.nonzero(ratio > threshold)[0]
    if len(above) == 0:
        return None
    return int(above[0])


ratio = sta_lta(strength[4], 50, 500)
plt.plot(seconds, ratio, lw=0.8)
plt.axhline(3, color="C1")
plt.axvline(p_index[4] / SAMPLE_RATE, color="k")
plt.xlabel("time (s)")
plt.ylabel("short average / long average")
plt.title("STA/LTA on 1 trace — black is the analyst's P, orange is the trigger level")
plt.show()

### Predict before you run

That is the standard picker, on a clear recording, landing on the arrival. Now the whole held-out
set: on what fraction of 741 recordings do you think it puts the pick within half a
second of the analyst's? Commit to a number before you run the next cell — change `my_guess` to
whatever you think, then run it.

In [ ]:
my_guess = 0.85


def sta_lta_score(short, long, threshold):
    """Fraction of test traces STA/LTA places within half a second of the analyst's pick."""
    hits = 0
    for i in np.nonzero(~is_train)[0]:
        pick = first_trigger(sta_lta(strength[i], short, long), threshold)
        if pick is not None and abs(pick - p_index[i]) <= 50:
            hits = hits + 1
    return hits / (~is_train).sum()


never = 0
on_the_s = 0
for i in np.nonzero(~is_train)[0]:
    pick = first_trigger(sta_lta(strength[i], 50, 500), 3)
    if pick is None:
        never = never + 1
    elif abs(pick - s_index[i]) <= 50:
        on_the_s = on_the_s + 1

print("you guessed:", my_guess)
print("textbook setting (0.5 s, 5 s, threshold 3):",
      round(sta_lta_score(50, 500, 3), 3))
print("  fraction that never triggered:  ", round(never / (~is_train).sum(), 3))
print("  fraction that hit the S instead:", round(on_the_s / (~is_train).sum(), 3))

### ✏️ Your turn 3

One setting is not a result. Run `sta_lta_score` on three more settings and print all three, so
that we know whether 0.453 is what STA/LTA can do or merely what this
particular setting does. Try these:

| short | long | threshold |
|---|---|---|
| 30 | 300 | 3 |
| 20 | 200 | 3 |
| 50 | 500 | 5 |

Then print the best of the four scores.

**Use these names**, because the self-check looks for them: `stalta_scores` for the list you
collect, and `best_stalta` for the best of the four.

(Each call is a Python loop over every sample of every held-out trace, so none of them is
instant.)

In [ ]:
# ← your answer here


assert len(stalta_scores) == 3, "three more settings were asked for, so this should hold three"
print("✓ the sweep — STA/LTA's best of four settings is",
      round(100 * best_stalta, 1), "%")

Tuned as well as four tries can tune it, the classical picker lands within half a second on
60.2% of held-out traces. It is not doing something silly. At the textbook
setting it never triggers at all on 27.0% of them — the noise was too loud for
the ratio to ever jump by three — and on another 18.5% it triggers on the S
instead of the P, having ignored a P that was too gentle to move the ratio.

The setting that came out best is also the one with the shortest windows, and that is not a
coincidence: a five-second long window has nothing to compare against until five seconds in, and
the setup cell told you some of these P arrivals come earlier than that.

That is the number to beat. Anything we build now has to beat 0.602, or we have
built decoration.

## Sliding a pattern-detector along the signal

Look again at what each of STA/LTA's two averages is. Take a small list of weights — one over the
window length, repeated — line it up against the samples ending at the current one, multiply and
add. Then move along one sample and do it again. That sliding-and-summing has a name: it is a
**convolution**. Slide a small pattern-detector along the signal. STA/LTA does it twice, with two
different window lengths, and divides the answers.

The small list of weights is the detector, and what it detects depends entirely on the numbers in
it. Put a step in the weights and it responds where the signal steps. `np.convolve` slides it for
you — lining the weights up in reverse, which is the difference between a convolution and a
sliding dot product, so the *first* half of the list is the half that lands on the most recent
samples.

In [ ]:
def onset_filter(width):
    """Energy in the last `width` samples minus energy in the `width` samples before those."""
    return np.concatenate([np.ones(width) / width, -np.ones(width) / width])


response = np.convolve(strength[4] ** 2, onset_filter(10), mode="same")

plt.plot(seconds, response / np.abs(response).max(), lw=0.8)
plt.axvline(p_index[4] / SAMPLE_RATE, color="k")
plt.axvline(s_index[4] / SAMPLE_RATE, color="C1")
plt.xlabel("time (s)")
plt.ylabel("filter response (scaled to its own peak)")
plt.title("a 20-sample onset detector on 1 trace — black P, orange S")
plt.show()

### ✏️ Your turn 4

Score that hand-made detector the way you scored STA/LTA. Loop over the held-out traces, convolve
`strength[i] ** 2` with `onset_filter(10)`, and collect `response.argmax()` — the sample where the
detector responds most — into a list, one entry per held-out trace.

Then print two fractions: how often that pick lands within 50 samples of `p_index[i]`, and —
because the last figure hints at where it actually goes — how often it lands within 50 samples of
`s_index[i]`.

**Use these names**, because the self-check looks for them: `hand_picks` for the list and
`hand_accuracy` for the first fraction.

In [ ]:
# ← your answer here


assert len(hand_picks) == (~is_train).sum(), "one pick per held-out trace, and none of the rest"
print("✓ the hand-made detector — it finds the P on",
      round(100 * hand_accuracy, 1), "% of held-out traces")

Worse than STA/LTA, and worse in an informative way: it lands near the **S** on
59.2% of traces. The numbers we chose describe *the biggest jump in energy*, and
in a seismogram the biggest jump in energy is the S.

We could keep guessing weights. A better detector might be longer, or shorter, or shaped
differently, or three detectors combined; there is no reason to think a human is good at choosing
twenty numbers. So stop choosing them.

## Letting the machine choose the numbers

**A stack of the logistic regressions you already know.** That is all a neural network is. Take the piece it stacks first.
**One weighted sum, squashed into an answer — a single logistic regression, and the piece a network is built out of.** That is a **perceptron**. Put several of them side by side, feed their
outputs into another row of them, and you have a stack.

Between the rows sits one more step. **The bend between two layers. Without it, a stack of straight lines is still one straight line.** That is the **activation**, and it
is what makes stacking worth anything: without it, a weighted sum of weighted sums is still a
weighted sum, however many rows you use. Here are two stacks of identical shape, one with the bend
and one without.

In [ ]:
grid = torch.linspace(-3, 3, 200).reshape(-1, 1)
torch.manual_seed(1)
flat = nn.Sequential(nn.Linear(1, 8), nn.Linear(8, 1))
torch.manual_seed(1)
bent = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))

plt.plot(grid.numpy(), flat(grid).detach().numpy(), label="2 layers, no activation")
plt.plot(grid.numpy(), bent(grid).detach().numpy(), label="2 layers, with ReLU")
plt.xlabel("input")
plt.ylabel("output")
plt.title("what 8 hidden units can draw, with and without an activation")
plt.legend()
plt.show()

Three more words and we can build one.

**The one number the network is trying to make small.** That is the **loss**, and ours is the same one you used to fit a straight
line: the average squared miss between what the network says and what we wanted.

**Roll downhill on the error surface.** That is **gradient descent** — the loss depends on every weight in
the network, PyTorch works out which way each weight would have to move to make the loss smaller,
and every weight takes a small step that way. **One pass through all of the training data.** That is an **epoch**, and
training is doing it again and again.

The last thing to decide is what "what we wanted" means. We are not asking for a number. We are
asking the network, at each of the 2,048 samples, *how much does this look like the
P arrival* — so the answer we train it towards is a bump centred on the analyst's mark, and the
pick we read back out is wherever the network's answer is highest.

In [ ]:
def make_target(pick_index, sigma):
    """A bump centred on each trace's P arrival: what we want the network to output."""
    sample = np.arange(2048)
    target = np.zeros((len(pick_index), 2048), dtype="float32")
    for i in range(len(pick_index)):
        target[i] = np.exp(-(sample - pick_index[i]) ** 2 / (2 * sigma ** 2))
    return target


plt.plot(seconds, waveform[4][2] / np.abs(waveform[4][2]).max(), lw=0.6,
         label="up-down, scaled")
plt.plot(seconds, make_target(p_index[4:4 + 1], 20)[0], label="what we want back")
plt.xlabel("time (s)")
plt.ylabel("scaled to 1")
plt.title("1 trace and its training target — a bump of width 20 samples on the P")
plt.legend()
plt.show()

Now the network. `nn.Conv1d(3, 8, 7)` is eight pattern-detectors, each 7 samples long, each
looking at all 3 rows at once — the same sliding operation as before, except that the numbers
inside are what training will choose. `stride=4` slides in steps of 4 instead of 1, which makes
the signal four times shorter and lets the next layer's 7 samples cover four times as much time;
`padding=3` keeps each layer's output the same length as its input, and `nn.Upsample` stretches
the whole thing back out at the end, so the answer is one number per original sample.

In PyTorch a model is an `nn.Module` — a box that holds the numbers to be learned and knows how
to run them. `nn.Sequential` is the simplest one there is: hand it layers, and it runs them in
order.

In [ ]:
def make_picker():
    """Five convolution layers: squeeze the trace down to a summary, then stretch it back out."""
    return nn.Sequential(
        nn.Conv1d(3, 8, 7, stride=4, padding=3), nn.ReLU(),
        nn.Conv1d(8, 16, 7, stride=4, padding=3), nn.ReLU(),
        nn.Conv1d(16, 16, 7, padding=3), nn.ReLU(),
        nn.Upsample(scale_factor=4), nn.Conv1d(16, 8, 7, padding=3), nn.ReLU(),
        nn.Upsample(scale_factor=4), nn.Conv1d(8, 1, 7, padding=3))


x_train = torch.tensor(waveform[is_train])
x_test = torch.tensor(waveform[~is_train])
p_test = p_index[~is_train]

print("weights to learn:", sum(w.numel() for w in make_picker().parameters()))

In [ ]:
def picks_from(model, x):
    """Where the network says the P is: the sample with the highest output."""
    return model(x).squeeze(1).detach().numpy().argmax(axis=1)


def within_half_second(picks, truth):
    """Fraction of picks landing within half a second of the analyst's pick."""
    return (np.abs(picks - truth) <= 50).mean()


def train_picker(sigma=20, epochs=25):
    """Train the picker; hand back the model, and the loss and test score after every epoch."""
    torch.manual_seed(0)
    model = make_picker()
    optimiser = torch.optim.Adam(model.parameters(), lr=0.005)
    loss_function = nn.MSELoss()
    y_train = torch.tensor(make_target(p_index[is_train], sigma))
    losses, scores = [], []
    for epoch in range(epochs):
        order = torch.randperm(len(x_train))
        total = 0.0
        for start in range(0, len(x_train), 32):
            batch = order[start:start + 32]
            loss = loss_function(model(x_train[batch]).squeeze(1), y_train[batch])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total = total + loss.item() * len(batch)
        losses.append(total / len(x_train))
        scores.append(within_half_second(picks_from(model, x_test), p_test))
    return model, losses, scores


picker, losses, scores = train_picker()
print("trained for", len(losses), "epochs")

## Watching it learn

Two lines, on two axes because they are in different units. On the left, the loss on the data the
network trained on. On the right, the fraction of **held-out** traces it picks within half a
second — recordings of earthquakes it has never seen. **Watch two lines. When training keeps
falling and test turns up, stop.**

The left line is what gradient descent is directly pushing down, so it should fall smoothly. The
right line is the one we actually care about, and nothing is pushing on it at all.

In [ ]:
epochs = np.arange(1, len(losses) + 1)

plt.figure(figsize=(9, 3.5))                    # two panels side by side need a wider figure
plt.subplot(1, 2, 1)
plt.plot(epochs, losses)
plt.xlabel("epoch")
plt.ylabel("training loss")

plt.subplot(1, 2, 2)
plt.plot(epochs, scores)
plt.xlabel("epoch")
plt.ylabel("held-out traces within 0.5 s")
plt.suptitle("learning curve — 1,759 training and 741 held-out recordings")
plt.show()

print("after epoch 10 the held-out line still wanders by",
      round(max(scores[10:]) - min(scores[10:]), 3), "from epoch to epoch")

### ✏️ Your turn 5

Report the result properly, so it can be set beside 0.602.

Get the network's picks on the held-out traces with `picks_from(picker, x_test)`, then print
three things: the fraction within half a second, the **median** error in seconds, and the epoch
at which the held-out score was highest — `scores` is an ordinary list, so `np.argmax(scores)`
gives its position and epochs are counted from 1.

**Use these names**, because the self-check looks for them: `net_picks` and `net_accuracy`.

In [ ]:
# ← your answer here


assert len(net_picks) == len(p_test), "one pick per held-out trace"
print("✓ the network —", round(100 * net_accuracy, 1), "% of held-out traces within 0.5 s,",
      "median error", round(np.median(np.abs(net_picks - p_test)) / SAMPLE_RATE, 3), "s")

Two of the held-out recordings, drawn: the one the network places most accurately and the one it
places worst. Behind the network's answer in each panel is the liveliest of that recording's three
rows, and the black line is where the analyst put the P.

The lower panel is not a plotting mistake. The cell prints the standard deviation of each of that
recording's three rows, and all three are zero — every channel is stuck at the end of its range,
so there is no ground motion in the file to find. An archive of real instruments contains records
like that, and no picker can be blamed for them.

In [ ]:
closest = np.abs(net_picks - p_test).argmin()
furthest = np.abs(net_picks - p_test).argmax()

plt.figure(figsize=(7, 5))                      # two stacked traces need the extra height
for panel, which in [(1, closest), (2, furthest)]:
    liveliest = x_test[which].numpy().std(axis=1).argmax()   # the row with the most in it
    plt.subplot(2, 1, panel)
    plt.plot(seconds, x_test[which][liveliest].numpy() / 10, lw=0.5, label="ground motion, scaled")
    plt.plot(seconds, picker(x_test[[which]]).squeeze().detach().numpy(), lw=1,
             label="the network's answer")
    plt.axvline(p_test[which] / SAMPLE_RATE, color="k")
    plt.ylabel("scaled to 1")
    if panel == 1:
        plt.legend()
plt.xlabel("time (s)")
plt.suptitle("the network on its closest and its furthest of {} held-out recordings"
             .format(len(p_test)))
plt.show()

print("standard deviation of the three rows, worst recording:",
      x_test[furthest].numpy().std(axis=1).round(3))

## The question, answered

Yes, and better than the method it replaces: on 741 held-out recordings from
earthquakes it had never seen, a network of 3,857 weights put the P arrival within
half a second on 88.9% of them, against 60.2% for the best of four
STA/LTA settings — and it did it by learning the shape of an onset, on traces where the loudest
moment is a different wave entirely.

## Week 13 summary

**The question.** Can a machine hear an earthquake?

### What to remember

| | |
|---|---|
| **1** | A network learns waveform shape once amplitude is taken away from it. |
| **2** | Beat the classical method first, or the network is decoration. |
| **3** | Labels have a ceiling, and a model already at that ceiling cannot be improved by more training. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Perceptron** | One weighted sum, squashed into an answer — a single logistic regression, and the piece a network is built out of. |
| **Activation** | The bend between two layers. Without it, a stack of straight lines is still one straight line. |
| **Loss** | The one number the network is trying to make small. |
| **Epoch** | One pass through all of the training data. |
| **Neural network** | A stack of the logistic regressions you already know. |
| **Gradient descent** | Roll downhill on the error surface. |
| **1-D CNN** | Slide a small pattern-detector along the signal. |

### Code you met this week

| Function | What it does |
|---|---|
| `np.clip(a, low, high)` | trim every value into a range — here, cutting off instrument spikes |
| `np.convolve(signal, weights, mode="same")` | slide a small pattern-detector along a signal |
| `np.unique(a)` | each different value in an array, once |
| `array.std(axis=n)` | the typical size of the wiggles, one number per row |
| `torch.tensor(array)` | hand an array to PyTorch so it can be learned from |
| `torch.manual_seed(n)` | fix the random start, so a training run repeats |
| `nn.Sequential(layers)` | a model that runs the layers you hand it, in order |
| `nn.Conv1d(in, out, width)` | a row of pattern-detectors slid along the signal, with weights the training chooses |
| `nn.ReLU()` | the activation — the bend that makes a stack more than one straight line |
| `nn.Linear(in, out)` | one layer of weighted sums — a row of perceptrons |
| `nn.MSELoss()` | the loss: the average squared miss, the same one that fits a straight line |
| `torch.optim.Adam(model.parameters(), lr=)` | the thing that rolls the weights downhill |
| `loss.backward() / optimiser.step() / optimiser.zero_grad()` | work out which way each weight should move, move them, then clear the slate |
| `nn.Upsample(scale_factor=n)` | stretch a shortened signal back to its original length |

## Homework

Three parts. The first asks something about the class result that class did not ask; the second
asks you to decide something and defend the decision; the third settles an argument class left
open.

Two of the three train a network from scratch, so start them and let them run. If you restarted
the kernel since class, run the checkpoint cell first — it rebuilds the trained picker without
printing anything, and it is the slow cell rather than a free one.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
picker, losses, scores = train_picker()
net_picks = picks_from(picker, x_test)

### ✏️ Your turn 6

The two pickers were scored on the same held-out recordings, and those recordings are not equally
easy: `snr` says how many times louder the earthquake is than the background on each one. An
average over all of them hides that. So where does the network's advantage actually come from —
is it ahead everywhere, or only where the signal is faint?

Split the held-out recordings in half at the median of their `snr` and report four numbers: the
network's accuracy and STA/LTA's on the quiet half, and both again on the loud half. Score
STA/LTA at its best setting from Your turn 3 — short 30,
long 300, threshold
3 — the way `sta_lta_score` does, except keeping one
True/False per recording instead of one total.

Finish by printing how much accuracy each method loses going from the loud half to the quiet half.

**Use these names**, because the self-check looks for them: `classic_ok` and `network_ok`, each
one True/False per held-out recording, and `quiet` for the mask picking out the quiet half.

In [ ]:
# ← your answer here


assert len(classic_ok) == len(network_ok) == len(quiet),     "one True/False per held-out recording, for both methods"
print("✓ by loudness — on the quiet half the network scores",
      round(network_ok[quiet].mean(), 3), "against STA/LTA's",
      round(classic_ok[quiet].mean(), 3))

### ✏️ Your turn 7

The bump we trained towards was 20 samples wide, and nobody made us choose 20. A narrow bump asks
the network for a precise answer and gives it almost nothing to aim at; a wide one is easy to hit
and blurry when you read the peak back off it.

**Pick one and defend it: `sigma=5` (0.05 s) or `sigma=40` (0.4 s).** Retrain with
`train_picker(sigma=...)`, and print the final held-out fraction within half a second, the median
error in seconds, and the final training loss. Then print the same three numbers for the network
you already trained at `sigma=20`, so the comparison is on the page.

Say in the same cell, as a printed line, which you chose and why.

**Use these names**, because the self-check looks for them: `my_sigma` and `my_scores`.

(Careful with the losses: a narrow bump is mostly zeros, so its loss is a smaller number even when
the network is doing worse. Losses computed against different targets are not comparable.)

In [ ]:
# ← your answer here


assert my_sigma in (5, 40), "the question offers 5 or 40; pick one of them"
assert len(my_scores) == len(scores), "same number of epochs, so only the label width changed"
print("✓ the label width — sigma", my_sigma, "scores",
      round(my_scores[-1], 3), "against", round(scores[-1], 3), "at sigma 20")

### ✏️ Your turn 8

In class the held-out score was still at its highest on the very last epoch, which leaves an
obvious doubt: perhaps the picker was simply not trained for long enough.

Settle it. Run `train_picker(epochs=60)` — this is the slow one — and print four numbers from
what it hands back: the training loss and the held-out score after epoch 25, and both again after
epoch 60. Print the best held-out score of the whole run and the epoch it came at as well. (A
list counts from 0, so epoch 25 is at position 24.)

Then, in the **markdown** cell below the code cell, answer in two or three sentences **using your
own four numbers**: over
those 35 extra epochs, what happened to the loss, what happened to the held-out score, and what
does the pair of answers say about where this picker's remaining error is coming from? Your answer
has to name a number that would have to change before the picker could do better.

**Use these names**, because the self-check looks for them: `long_losses` and `long_scores`.

In [ ]:
# ← your answer here


assert len(long_scores) == 60, "60 epochs were asked for"
print("✓ more training — the loss fell by a factor of",
      round(long_losses[0] / long_losses[59], 1), "while the held-out score moved from",
      round(long_scores[24], 3), "to", round(long_scores[59], 3))

*(Double-click this cell and replace this line with your answer.)*